<a href="https://colab.research.google.com/github/AyaanMahajan/econ3916-lab13-multivariate-ols/blob/main/Lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ECON 3916 Lab 13
## Multivariate OLS — What "Controlling For" Actually Does

This lab looks at what changes when another variable is added to a regression and why some controls help while others can make the estimate worse

## Setup

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(13)

# Part 1: Omitted Variable Bias, With the Truth Known

In [2]:
n = 5000

ability = rng.normal(0, 1, n)
experience = 8 + 2 * ability + rng.normal(0, 2, n)
training = 2 + 0.7 * ability + rng.normal(0, 1, n)

# the true training effect is 3
wage = 20 + 3 * training + 2 * experience + rng.normal(0, 3, n)

payroll = pd.DataFrame({
    "wage": wage,
    "training": training,
    "experience": experience,
    "ability": ability
})

payroll.head()

,wage,training,experience,ability
0,45.852140,2.587278,9.595862,1.826757
1,17.400880,0.863461,1.257101,-3.078332
2,49.653416,2.631609,8.121700,0.958064
3,38.724649,1.018901,9.391013,0.069637
4,45.100388,2.334387,8.935547,1.318250


In [3]:
bivariate = smf.ols("wage ~ training", data=payroll).fit()
controlled = smf.ols("wage ~ training + experience", data=payroll).fit()

print(f"Bivariate training coefficient: {bivariate.params['training']:.3f}")
print(f"Controlled training coefficient: {controlled.params['training']:.3f}")
print("True training effect: 3.000")

Bivariate training coefficient: 4.885
Controlled training coefficient: 3.015
True training effect: 3.000


The bivariate coefficient is about **4.89** even though the true effect is only **3** and that happens because training and experience move together while experience also raises wages

Once experience is controlled for the training coefficient falls to about **3.01** which is basically the true effect

Before running the second regression I would expect the omitted variable bias to be positive because training is positively related to experience and experience has a positive effect on wages

# Part 2: Frisch-Waugh-Lovell

In [4]:
# remove the part of wage explained by experience
wage_resid = smf.ols(
    "wage ~ experience",
    data=payroll
).fit().resid

# remove the part of training explained by experience
training_resid = smf.ols(
    "training ~ experience",
    data=payroll
).fit().resid

fwl_fit = sm.OLS(wage_resid, training_resid).fit()

print(f"FWL training coefficient: {fwl_fit.params.iloc[0]:.3f}")
print(f"Multivariate training coefficient: {controlled.params['training']:.3f}")

FWL training coefficient: 3.015
Multivariate training coefficient: 3.015


The two coefficients are the same at about **3.01** and this is what controlling for experience actually does

It removes the part of training connected to experience and it removes the part of wages connected to experience and then it compares what is left

# Part 3: The Control That Creates Bias

In [5]:
# manager rating is caused by training and wage
# so it is not a good variable to control for here
payroll["manager_rating"] = (
    0.8 * payroll["training"]
    + 0.06 * payroll["wage"]
    + rng.normal(0, 1, n)
)

good_model = smf.ols(
    "wage ~ training + experience",
    data=payroll
).fit()

bad_model = smf.ols(
    "wage ~ training + experience + manager_rating",
    data=payroll
).fit()

print(f"True training effect: 3.000")
print(f"Good controls: {good_model.params['training']:.3f}")
print(f"After adding manager rating: {bad_model.params['training']:.3f}")

True training effect: 3.000
Good controls: 3.015
After adding manager rating: 2.509


The estimate is about **3.01** with experience in the model but it falls to about **2.51** after manager rating is added

Manager rating sounds like a useful control but it is partly created by training and wage so holding it fixed changes the comparison and pushes the training coefficient away from the truth

This is why I would not just control for every variable that is available

# Part 4: Interactions

In [6]:
female = rng.integers(0, 2, n)

wage_interaction = (
    20
    + 2.5 * training
    + 1.5 * female
    + 1.8 * training * female
    + 2 * experience
    + rng.normal(0, 3, n)
)

interaction_df = pd.DataFrame({
    "wage": wage_interaction,
    "training": training,
    "experience": experience,
    "female": female
})

interaction_fit = smf.ols(
    "wage ~ training * female + experience",
    data=interaction_df
).fit()

print(interaction_fit.params.round(3))

Intercept          20.114
training            2.502
female              1.438
training:female     1.769
experience          1.997
dtype: float64


For the group where `female = 0` the training effect is about **2.50**

The interaction coefficient is about **1.77** so for the group where `female = 1` the training effect is about **4.27** in total because I add the main training coefficient and the interaction coefficient

The interaction lets the slope change between the two groups instead of forcing the same training effect on everyone

# Extension: Choosing What Goes In

In [7]:
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing(as_frame=True)
housing_df = housing.frame.copy()

model_1 = smf.ols(
    "MedHouseVal ~ MedInc",
    data=housing_df
).fit()

model_2 = smf.ols(
    "MedHouseVal ~ MedInc + HouseAge + AveRooms + AveOccup",
    data=housing_df
).fit()

print(f"MedInc coefficient alone: {model_1.params['MedInc']:.4f}")
print(f"MedInc coefficient with controls: {model_2.params['MedInc']:.4f}")
print(f"R-squared alone: {model_1.rsquared:.3f}")
print(f"R-squared with controls: {model_2.rsquared:.3f}")

MedInc coefficient alone: 0.4179
MedInc coefficient with controls: 0.4433
R-squared alone: 0.473
R-squared with controls: 0.514


The MedInc coefficient changes after other housing variables are added and that is normal because the meaning of the coefficient has changed

The first model compares places with different incomes overall while the second compares places with different incomes while holding the other included housing variables fixed

A bigger R-squared does not automatically mean the second model gives a better causal estimate because that depends on whether the controls belong in the model for the question I am asking

# Challenge: Sign the Bias Before You Run It

In [8]:
rng_challenge = np.random.default_rng(21)

ability_2 = rng_challenge.normal(0, 1, n)
education = 12 + 1.2 * ability_2 + rng_challenge.normal(0, 1.5, n)

# true education effect is 2
wage_2 = (
    15
    + 2 * education
    + 4 * ability_2
    + rng_challenge.normal(0, 3, n)
)

challenge_df = pd.DataFrame({
    "wage": wage_2,
    "education": education,
    "ability": ability_2
})

naive_education = smf.ols(
    "wage ~ education",
    data=challenge_df
).fit()

controlled_education = smf.ols(
    "wage ~ education + ability",
    data=challenge_df
).fit()

print("Prediction before running: positive omitted variable bias")
print(f"Naive education coefficient: {naive_education.params['education']:.3f}")
print(f"Controlled education coefficient: {controlled_education.params['education']:.3f}")
print("True education effect: 2.000")

Prediction before running: positive omitted variable bias
Naive education coefficient: 3.289
Controlled education coefficient: 1.999
True education effect: 2.000


I predicted positive bias before running the regression because ability raises education and it also raises wages

The result matches that prediction because the education coefficient is about **3.31** when ability is left out and about **2.01** after ability is controlled for which is almost exactly the true effect of **2**

# AI-Assisted Expansion: Specification Table

This is the section where the assignment specifically asks for the model-assisted specification check

## Prompt version 1

Create a short regression specification table for my payroll example and compare a bivariate model with one that controls for experience and one that also controls for manager rating

## Prompt version 2

Create a short regression specification table for my payroll example but make the table focus on the training coefficient and explain that experience is a valid control in this simulation while manager rating is created by both training and wage so adding it can create bias

## Model response

| Specification | Variables included | Training coefficient | What it shows |
|---|---|---:|---|
| 1 | Training | about 4.89 | Training picks up part of the experience effect |
| 2 | Training + Experience | about 3.01 | The estimate is very close to the true effect of 3 |
| 3 | Training + Experience + Manager rating | about 2.51 | Adding manager rating moves the estimate away from the truth |

## My check

The table matches the results from my regressions and the main correction from the first prompt was making the role of each control clear

Experience belongs in the model because it affects wages and is related to training but manager rating should not be added just because it looks useful since it is partly caused by the variables in the relationship I am studying

# README Draft

# Lab 13: Multivariate OLS

## Objective

I used a simulated payroll example to see what controlling for another variable actually does to a regression coefficient and I also checked what happens when a bad control is added

## Main results

The simple regression gave a training coefficient of about **4.89** even though the true effect was **3**

After controlling for experience the coefficient fell to about **3.01** and the Frisch-Waugh-Lovell residual regression gave the same answer

When manager rating was also controlled for the estimate moved away from the truth and fell to about **2.51** so adding more controls was not automatically better

The interaction model showed that the training slope can be different across groups and the challenge also showed positive omitted variable bias when ability was left out of an education and wage regression

## Main takeaway

A regression coefficient always depends on what else is in the model and controlling for something means comparing observations after removing the part explained by that control so the choice of controls has to come from the question and how the variables are related

# Submission

Save the completed notebook to GitHub and then upload the `.ipynb` file to Canvas with the repository URL in the Comments box